In [1]:
# ============================
# 0) Reproducibility & Setup
# ============================
import os, shutil, json, math, time, pathlib, warnings, random
import numpy as np
import tensorflow as tf
from google.colab import drive

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Mixed precision (comment out if you see NaNs on your GPU)
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
except Exception:
    pass

warnings.filterwarnings("ignore")

# ============================
# 1) Mount Google Drive
# ============================
drive.mount('/content/drive')

# Your existing split folders (already created)
base_dir = '/content/drive/My Drive'
output_base = '/content/drive/My Drive/Stroke_split'
train_dir = os.path.join(output_base, 'train')
val_dir   = os.path.join(output_base, 'val')
test_dir  = os.path.join(output_base, 'test')

# Output dirs
run_name = time.strftime("stroke_cls_%Y%m%d_%H%M%S")
out_dir = f"/content/drive/My Drive/Palsy Models/{run_name}"
os.makedirs(out_dir, exist_ok=True)
tb_dir = os.path.join(out_dir, "tb")
os.makedirs(tb_dir, exist_ok=True)

# ============================
# 2) Data Pipeline (tf.data)
# ============================
IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = ['No Stroke', 'Stroke']  # enforce stable order

def make_ds(dir_path, shuffle=True):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        dir_path, labels='inferred', label_mode='binary',
        class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH,
        shuffle=shuffle, seed=SEED
    )
    return ds

train_ds = make_ds(train_dir, shuffle=True)
val_ds   = make_ds(val_dir, shuffle=False)
test_ds  = make_ds(test_dir, shuffle=False)

# Normalization & Augmentations
norm = tf.keras.layers.Rescaling(1./255)

data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augment")

def prepare(ds, training=False):
    ds = ds.map(lambda x, y: (norm(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augment(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(1000, seed=SEED)
    return ds.cache().prefetch(AUTOTUNE)

train_ds_p = prepare(train_ds, training=True)
val_ds_p   = prepare(val_ds, training=False)
test_ds_p  = prepare(test_ds, training=False)

# ============================
# 3) Compute Class Weights (for imbalance)
# ============================
# Count labels from train_ds once
pos, neg = 0, 0
for _, y in train_ds.unbatch():
    if int(y.numpy()[0]) == 1:
        pos += 1
    else:
        neg += 1
total = pos + neg
class_weight = {
    0: total / (2.0 * neg + 1e-6),
    1: total / (2.0 * pos + 1e-6),
}
print("Class counts -> No Stroke:", neg, "Stroke:", pos)
print("Class weights:", class_weight)

# ============================
# 4) Build Trustworthy Model (Transfer Learning + MC Dropout)
# ============================
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import efficientnet

base = efficientnet.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,)
)
base.trainable = False  # 1st stage: freeze

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = inputs
x = norm(x)  # safety (also used in ds, harmless redundancy)
x = data_augment(x, training=True)  # ensure augmentation in-graph
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x, training=True)  # MC Dropout ON at inference when requested
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x, training=True)  # another MC dropout
outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)  # float32 head for stability
model = Model(inputs, outputs, name="StrokeClassifier_EffB0")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                       tf.keras.metrics.AUC(name='auc', curve='ROC'),
                       tf.keras.metrics.AUC(name='pr_auc', curve='PR')])

model.summary()

# ============================
# 5) Callbacks (BestModel, EarlyStop, LR schedule, Logs)
# ============================
ckpt_path = os.path.join(out_dir, "best_model.keras")
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_auc', mode='max',
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', mode='max',
        patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.TensorBoard(log_dir=tb_dir, histogram_freq=0)
]

# ============================
# 6) Train (Stage 1: head only)
# ============================
EPOCHS_1 = 8
hist1 = model.fit(
    train_ds_p, validation_data=val_ds_p,
    epochs=EPOCHS_1, class_weight=class_weight,
    callbacks=callbacks, verbose=1
)

# Unfreeze top of the backbone for fine-tuning
base.trainable = True
for layer in base.layers[:-40]:  # fine-tune last ~40 layers
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                       tf.keras.metrics.AUC(name='auc', curve='ROC'),
                       tf.keras.metrics.AUC(name='pr_auc', curve='PR')])

EPOCHS_2 = 20
hist2 = model.fit(
    train_ds_p, validation_data=val_ds_p,
    epochs=EPOCHS_2, class_weight=class_weight,
    callbacks=callbacks, verbose=1
)

# Load best checkpoint
best_model = tf.keras.models.load_model(ckpt_path)

# ============================
# 7) Eval: Metrics, Curves, Threshold Tuning
# ============================
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, \
                            confusion_matrix, classification_report, brier_score_loss

def collect(ds):
    y_true = []
    y_prob = []
    for x, y in ds:
        p = best_model.predict(x, verbose=0).ravel()
        y_prob.extend(p.tolist())
        y_true.extend(y.numpy().ravel().tolist())
    return np.array(y_true, dtype=np.float32), np.array(y_prob, dtype=np.float32)

y_val, p_val = collect(val_ds_p)
y_test, p_test = collect(test_ds_p)

# ROC & PR
fpr, tpr, thr = roc_curve(y_val, p_val)
roc_auc = auc(fpr, tpr)
prec, rec, thr_pr = precision_recall_curve(y_val, p_val)
ap = average_precision_score(y_val, p_val)

# Threshold: maximize F1 on val
def f1_at_thr(y, p, t=0.5):
    yhat = (p >= t).astype(int)
    tp = ((yhat==1) & (y==1)).sum()
    fp = ((yhat==1) & (y==0)).sum()
    fn = ((yhat==0) & (y==1)).sum()
    if tp==0 and (fp>0 or fn>0): return 0.0
    prec = tp / (tp+fp+1e-9)
    rec  = tp / (tp+fn+1e-9)
    return 2*prec*rec/(prec+rec+1e-9)

ths = np.linspace(0.1, 0.9, 81)
f1s = np.array([f1_at_thr(y_val, p_val, t) for t in ths])
best_t = ths[np.argmax(f1s)]
print(f"[VAL] ROC-AUC={roc_auc:.4f}, PR-AUC={ap:.4f}, Best-F1 thr={best_t:.3f}")

# Calibration: reliability curve + Brier score
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_val, p_val, n_bins=10, strategy='uniform')
brier = brier_score_loss(y_val, p_val)
print(f"[VAL] Brier score (lower better): {brier:.4f}")

# ============================
# 8) Temperature Scaling (calibration on val)
# ============================
T = tf.Variable(1.0, dtype=tf.float32, trainable=True)

@tf.function
def nll_with_temp(logits, y_true_bin):
    # logits are logit(p); Convert p->logit: log(p/(1-p))
    # We have probabilities; convert back to logits carefully
    eps = tf.constant(1e-7, tf.float32)
    p = tf.clip_by_value(logits, eps, 1.0-eps)
    logit = tf.math.log(p) - tf.math.log(1.0 - p)
    scaled = logit / T
    # sigmoid + BCE
    q = tf.sigmoid(scaled)
    y = tf.cast(y_true_bin, tf.float32)
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y, q))

opt = tf.keras.optimizers.Adam(1e-2)
for _ in range(200):
    with tf.GradientTape() as tape:
        loss = nll_with_temp(p_val, y_val)
    grads = tape.gradient(loss, [T])
    opt.apply_gradients(zip(grads, [T]))
T_val = float(T.numpy())
print(f"Calibrated temperature T={T_val:.3f}")

def apply_temp(p):  # apply calibrated temperature to probabilities
    p = np.clip(p, 1e-7, 1-1e-7)
    logit = np.log(p) - np.log(1-p)
    scaled = logit / T_val
    return 1.0/(1.0+np.exp(-scaled))

p_test_cal = apply_temp(p_test)

# Choose operating threshold on calibrated val set
p_val_cal = apply_temp(p_val)
f1s_cal = np.array([f1_at_thr(y_val, p_val_cal, t) for t in ths])
best_t_cal = ths[np.argmax(f1s_cal)]
print(f"[VAL] Best F1 threshold after calibration: {best_t_cal:.3f}")

# Final test metrics (calibrated)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
yhat_test = (p_test_cal >= best_t_cal).astype(int)
acc = accuracy_score(y_test, yhat_test)
prec_t, rec_t, f1_t, _ = precision_recall_fscore_support(y_test, yhat_test, average='binary')
cm = confusion_matrix(y_test, yhat_test)

print(f"\n[TEST] Accuracy: {acc*100:.2f}%")
print(f"[TEST] Precision: {prec_t*100:.2f}%")
print(f"[TEST] Recall: {rec_t*100:.2f}%")
print(f"[TEST] F1: {f1_t*100:.2f}%")

# Curves & Confusion Matrix
import seaborn as sns
plt.figure(figsize=(5,4)); plt.plot([0,1],[0,1],'--'); plt.plot(fpr,tpr); plt.title(f"ROC (AUC={roc_auc:.3f})"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.show()
plt.figure(figsize=(5,4)); plt.plot(rec,prec); plt.title(f"PR (AP={ap:.3f})"); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.show()
plt.figure(figsize=(5,4));
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False, square=True)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Test)"); plt.show()
plt.figure(figsize=(5,4));
plt.plot(prob_pred, prob_true, marker='o'); plt.plot([0,1],[0,1],'--')
plt.title(f"Reliability Curve (Brier={brier:.3f})"); plt.xlabel("Predicted prob"); plt.ylabel("Empirical prob"); plt.show()

# ============================
# 9) Grad-CAM for Explainability
# ============================
def grad_cam(img_tensor, model, last_conv_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-7)
    return heatmap.numpy()

# Pick a few test images
test_imgs, test_labels = next(iter(test_ds.take(1)))
test_imgs_norm = norm(test_imgs)
preds = best_model.predict(test_imgs_norm, verbose=0).ravel()

# Find the last conv layer name in EfficientNetB0
last_conv = None
for l in reversed(best_model.layers):
    if isinstance(l, tf.keras.layers.Conv2D):
        last_conv = l.name
        break
if last_conv is None:
    # EfficientNetB0 uses fused MBConv; fallback to last block
    for l in reversed(best_model.layers):
        if "top_activation" in l.name or "block7a" in l.name:
            last_conv = l.name
            break

import cv2
for i in range(min(6, test_imgs.shape[0])):
    img = test_imgs[i].numpy().astype('uint8')
    x = tf.expand_dims(test_imgs_norm[i], axis=0)
    heat = grad_cam(x, best_model, last_conv)
    heat = cv2.resize(heat, (IMG_SIZE[1], IMG_SIZE[0]))
    heat = np.uint8(255 * heat)
    heat = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    over = cv2.addWeighted(img, 0.6, heat, 0.4, 0)
    plt.figure(figsize=(6,3))
    plt.subplot(1,2,1); plt.imshow(img); plt.axis('off'); plt.title(f"True: {int(test_labels[i].numpy()[0])}")
    plt.subplot(1,2,2); plt.imshow(over); plt.axis('off'); plt.title(f"Pred p(stroke)={preds[i]:.2f}")
    plt.show()

# ============================
# 10) Uncertainty (MC Dropout + Abstention)
# ============================
def mc_predict(model, batch, T=20):
    # Run forward passes with dropout active
    preds = []
    for _ in range(T):
        p = model(batch, training=True).numpy().ravel()  # dropout ON
        preds.append(p)
    preds = np.stack(preds, axis=0)
    return preds.mean(0), preds.std(0)

mean_p, std_p = mc_predict(best_model, test_imgs_norm, T=30)
# Abstain if: (1) calibrated prob near 0.5 OR (2) uncertainty high
cal_mean = apply_temp(mean_p)
uncertainty = std_p
abstain_mask = (np.abs(cal_mean - 0.5) < 0.1) | (uncertainty > 0.06)
print(f"Abstain on {abstain_mask.sum()} / {abstain_mask.size} samples (~{100*abstain_mask.mean():.1f}%)")

# ============================
# 11) Save: Model (.keras), Label Map, Model Card
# ============================
final_model_path = os.path.join(out_dir, "stroke_classifier_calibrated.keras")
best_model.save(final_model_path)
with open(os.path.join(out_dir, "labels.json"), "w") as f:
    json.dump({"0":"No Stroke", "1":"Stroke"}, f, indent=2)

# Minimal Model Card (customize with clinical details!)
card = f"""# Model Card: Stroke Image Classifier
- Date: {time.strftime('%Y-%m-%d')}
- Architecture: EfficientNetB0 + custom head (MC Dropout)
- Input size: {IMG_SIZE}
- Training data: {train_dir}
- Validation data: {val_dir}
- Test data: {test_dir}
- Class weights: {class_weight}
- Metrics (VAL): ROC-AUC={roc_auc:.3f}, PR-AP={ap:.3f}, Brier={brier:.3f}
- Threshold (post-calibration): {best_t_cal:.3f}
- Uncertainty: MC Dropout (T=30); abstain if |p-0.5|<0.1 or std>0.06
- Explainability: Grad-CAM over last conv block
- Known limits: performance may vary across scanners/sites; requires 224x224 RGB; not a diagnostic tool.
- Intended use: clinician-assistive triage, not autonomous diagnosis.
"""
with open(os.path.join(out_dir, "MODEL_CARD.md"), "w") as f:
    f.write(card)

print("\n✅ Saved:")
print(" - Best checkpoint:", ckpt_path)
print(" - Final model (calibrated):", final_model_path)
print(" - Label map:", os.path.join(out_dir, "labels.json"))
print(" - Model Card:", os.path.join(out_dir, "MODEL_CARD.md"))
print(" - TensorBoard logs:", tb_dir)

# ============================
# 12) (Optional) Subgroup hooks for bias audits
# ============================
# If you have metadata (e.g., site, device, age, sex), attach per-image attributes and compute metrics per subgroup.
# Placeholder structure:
# subgroup = np.array([...])  # same length as y_test
# for g in np.unique(subgroup):
#     idx = subgroup == g
#     print(g, precision_recall_fscore_support(y_test[idx], yhat_test[idx], average='binary'))


ValueError: mount failed